# Tiền xử lý FS1 (RFM) — chính sách P-keep

Chỉ tiền xử lý: raw → làm sạch có đối soát → bảng RFM theo khách. Chưa log, chưa scale, chưa mô hình hóa. Khác baseline: **giữ** dòng trùng và mã dịch vụ, **tách** chứng từ C sang bảng riêng thay vì xóa, không xóa dòng ngoại lai. Cấu hình thử nghiệm, chưa phải quyết định chính thức của nhóm.

## Params

In [1]:
REF_DATE = "2011-12-10"          # ngày tham chiếu Recency (ngày sau ngày cuối của dữ liệu)
WINDOW_START = "2010-12-01"
RAW_SHA256 = "43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d"
SERVICE_CODES = ["POST", "D", "M", "m", "C2", "DOT", "BANK CHARGES", "PADS", "S", "B", "CRUK", "AMAZONFEE"]
FEATURE_FS1 = ["Recency", "Frequency", "Monetary"]

## import thư viện

In [2]:
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import skew

Path("data").mkdir(exist_ok=True)

## Đọc raw và kiểm tra phiên bản

In [3]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/Online Retail.xlsx").exists())
raw_path = ROOT / "data/raw/Online Retail.xlsx"
assert hashlib.sha256(raw_path.read_bytes()).hexdigest() == RAW_SHA256, "Raw không đúng phiên bản dự án"

raw = pd.read_excel(raw_path, sheet_name="Online Retail",
                    dtype={"InvoiceNo": "string", "StockCode": "string", "CustomerID": "string"})
raw["Value"] = raw["Quantity"] * raw["UnitPrice"]
print("Raw:", raw.shape[0], "dòng ×", raw.shape[1] - 1, "cột gốc")

Raw: 541909 dòng × 8 cột gốc


**Nhận xét:** Raw đúng phiên bản (SHA-256 khớp), 541.909 dòng × 8 cột gốc. Cột `Value` = Quantity × UnitPrice được thêm để tính Monetary và đối soát giá trị.

## Làm sạch P-keep và đối soát số dòng

In [4]:
def snap(label, d, prev):
    return {"Bước": label, "Dòng còn lại": len(d), "Dòng bị loại/tách": prev - len(d),
            "% so với raw": round(len(d) / len(raw) * 100, 2), "Hóa đơn": d["InvoiceNo"].nunique(),
            "Khách có mã": d["CustomerID"].nunique(), "Giá trị (GBP)": round(d["Value"].sum(), 2)}

has_id = raw[raw["CustomerID"].notna()]
is_c = has_id["InvoiceNo"].str.startswith("C")
returns = has_id[is_c]                                   # giữ ở bảng riêng (dành cho feature trả hàng)
rest = has_id[~is_c]
purchases = rest[(rest["Quantity"] > 0) & (rest["UnitPrice"] > 0)]
assert purchases["InvoiceDate"].ge(WINDOW_START).all() and purchases["InvoiceDate"].lt(REF_DATE).all(), "Dòng mua ngoài cửa sổ"

rows, prev = [snap("Raw", raw, len(raw))], len(raw)
for label, d in [("Loại thiếu CustomerID", has_id),
                 ("Tách chứng từ C sang bảng trả hàng", rest),
                 ("Loại Quantity/UnitPrice ≤ 0 (không C)", purchases)]:
    rows.append(snap(label, d, prev)); prev = len(d)
accounting = pd.DataFrame(rows)
display(accounting)

no_buy = set(raw["CustomerID"].dropna()) - set(purchases["CustomerID"])
only_c = no_buy & set(returns["CustomerID"])
n_dup = int(purchases.duplicated().sum())
n_svc = int(purchases["StockCode"].isin(SERVICE_CODES).sum())
print("Khách có mã:", raw["CustomerID"].nunique(), "| có mua:", purchases["CustomerID"].nunique(),
      "| không có dòng mua:", len(no_buy), f"(chỉ có chứng từ C: {len(only_c)}; chỉ có dòng Qty/Price ≤ 0: {len(no_buy - only_c)})")
print("Giữ lại: dòng trùng hoàn toàn =", n_dup, "| dòng mã dịch vụ =", n_svc, "| dòng chứng từ C (bảng riêng) =", len(returns))

,Bước,Dòng còn lại,Dòng bị loại/tách,% so với raw,Hóa đơn,Khách có mã,Giá trị (GBP)
0,Raw,541909,0,100.00,25900,4372,9747747.93
1,Loại thiếu CustomerID,406829,135080,75.07,22190,4372,8300065.81
2,Tách chứng từ C sang bảng trả hàng,397924,8905,73.43,18536,4339,8911407.90
3,Loại Quantity/UnitPrice ≤ 0 (không C),397884,40,73.42,18532,4338,8911407.90


Khách có mã: 4372 | có mua: 4338 | không có dòng mua: 34 (chỉ có chứng từ C: 33; chỉ có dòng Qty/Price ≤ 0: 1)
Giữ lại: dòng trùng hoàn toàn = 5192 | dòng mã dịch vụ = 1547 | dòng chứng từ C (bảng riêng) = 8905


**Nhận xét:**

* Phần mất nhiều nhất là **135.080 dòng thiếu CustomerID (24,93% raw)**. Không gán được cho khách nào nên không khôi phục được; đây là nguồn chính của việc "xóa nhiều", không phải quy tắc làm sạch.
* 8.905 dòng chứng từ C được **tách sang bảng riêng**, không xóa (dành cho feature trả hàng FS3). Chỉ loại 40 dòng Quantity/UnitPrice ≤ 0 không phải chứng từ C, giá trị 0.
* Giữ lại 397.884 dòng mua (73,42% raw), nhiều hơn baseline (392.692) đúng 5.192 dòng trùng hoàn toàn. Mã dịch vụ (1.547 dòng) cũng được giữ.
* Số khách có mua là **4.338, bằng baseline**. 34 khách còn lại không có dòng mua nào hợp lệ (33 chỉ có chứng từ C, 1 chỉ có dòng Quantity/UnitPrice ≤ 0) nên không thể tính RFM.

## Tạo feature FS1 (RFM)

In [5]:
def build_rfm(p):
    g = p.groupby("CustomerID")
    out = pd.DataFrame({
        "Recency": (pd.Timestamp(REF_DATE) - g["InvoiceDate"].max().dt.normalize()).dt.days,
        "Frequency": g["InvoiceNo"].nunique(),
        "Monetary": g["Value"].sum().round(2),
    })
    return out.reset_index()

rfm = build_rfm(purchases)
assert rfm["CustomerID"].is_unique and len(rfm) == purchases["CustomerID"].nunique()
assert (rfm["Recency"] >= 1).all() and (rfm["Frequency"] >= 1).all() and (rfm["Monetary"] > 0).all()
assert rfm["Frequency"].sum() == purchases["InvoiceNo"].nunique()      # mỗi hóa đơn thuộc đúng một khách
assert np.isclose(rfm["Monetary"].sum(), purchases["Value"].sum(), atol=0.5)
assert not rfm[FEATURE_FS1].isna().any().any()
rfm.to_csv("data/customer_features.csv", index=False)

summary = pd.DataFrame({
    "Min": rfm[FEATURE_FS1].min(), "Median": rfm[FEATURE_FS1].median(), "Mean": rfm[FEATURE_FS1].mean(),
    "Max": rfm[FEATURE_FS1].max(),
    "Skew thô": [skew(rfm[c]) for c in FEATURE_FS1],
    "Skew log1p": [skew(np.log1p(rfm[c])) for c in FEATURE_FS1],
    "Thiếu": rfm[FEATURE_FS1].isna().sum()})
print("Số khách:", len(rfm))
display(summary.round(2))

Số khách: 4338


,Min,Median,Mean,Max,Skew thô,Skew log1p,Thiếu
Recency,1.00,51.00,93.06,374.00,1.25,-0.33,0
Frequency,1.00,2.00,4.27,209.00,12.06,1.21,0
Monetary,3.75,674.48,2054.27,280206.02,19.32,0.39,0


**Nhận xét:**

* 4.338 khách, không thiếu giá trị. Recency 1–374 ngày, Frequency trung vị 2 (tối đa 209), Monetary trung vị 674,48 nhưng trung bình 2.054 và tối đa 280.206.
* Độ lệch thô rất cao (Recency 1,25; Frequency 12,06; Monetary 19,32); sau `log1p` còn −0,33; 1,21; 0,39. Frequency vẫn lệch phải vì nhiều khách chỉ mua một lần. Phép log + scale chưa áp dụng ở bước này, để dành cho bước mô hình.
* Ngoại lai được giữ nguyên, không xóa dòng.

## Đối chiếu với chính sách baseline (mức dữ liệu)

In [6]:
# Baseline = cùng quy trình nhưng xóa dòng trùng hoàn toàn
rfm_b = build_rfm(purchases.drop_duplicates())
assert (rfm_b["CustomerID"].values == rfm["CustomerID"].values).all()
diff = {
    "Số khách": (len(rfm_b), len(rfm)),
    "Tổng Monetary (GBP)": (round(rfm_b["Monetary"].sum(), 2), round(rfm["Monetary"].sum(), 2)),
    "Khách có Monetary khác": ("", int((rfm_b["Monetary"] != rfm["Monetary"]).sum())),
    "Chênh Monetary lớn nhất (GBP)": ("", round(float((rfm["Monetary"] - rfm_b["Monetary"]).abs().max()), 2)),
    "Khách có Recency khác": ("", int((rfm_b["Recency"] != rfm["Recency"]).sum())),
    "Khách có Frequency khác": ("", int((rfm_b["Frequency"] != rfm["Frequency"]).sum())),
}
compare = pd.DataFrame(diff, index=["Baseline-policy", "FS1"]).T
display(compare)

,Baseline-policy,FS1
Số khách,4338,4338
Tổng Monetary (GBP),8887208.89,8911407.9
Khách có Monetary khác,,951
Chênh Monetary lớn nhất (GBP),,666.54
Khách có Recency khác,,0
Khách có Frequency khác,,0


**Nhận xét:** So với chính sách baseline, FS1 có cùng 4.338 khách; Recency và Frequency **không đổi**. Chỉ 951 khách (có dòng trùng) có Monetary khác, lớn nhất 666,54 GBP; tổng chênh 24.199 GBP (khoảng 0,27%). Giữ dòng trùng là thay đổi rất nhỏ, càng cho thấy phần dữ liệu "mất" của baseline chủ yếu là dòng thiếu mã khách.

## Lưu nhật ký

In [7]:
from openpyxl import Workbook
from openpyxl.styles import Font

B = Font(name="Calibri", size=11, bold=True)
wb = Workbook(); ws = wb.active; ws.title = "TienXuLy_FS1"
ws["A1"] = "NHẬT KÝ TIỀN XỬ LÝ — FEATURE FS1 (RFM, chính sách P-keep)"; ws["A1"].font = Font(name="Calibri", size=15, bold=True)
ws.merge_cells("A1:J1"); ws.row_dimensions[1].height = 33
ws["A2"] = "Trạng thái"
ws["B2"] = "Đã chạy tiền xử lý và tạo feature; chưa mô hình hóa. Thử nghiệm, chưa phải quyết định chính thức của nhóm"
ws.merge_cells("B2:J2"); ws.row_dimensions[2].height = 24
for c, v in {"A4": "Thiết lập", "B4": "Giá trị", "D4": "Kết quả", "E4": "Giá trị"}.items():
    ws[c] = v; ws[c].font = B
ws.row_dimensions[4].height = 25
left = [("Feature", "Recency, Frequency, Monetary"), ("Nguồn", "Online Retail.xlsx (SHA-256 khớp)"),
        ("Chính sách", "P-keep"), ("Ngày tham chiếu", REF_DATE),
        ("Cửa sổ quan sát", f"[{WINDOW_START}, {REF_DATE})"),
        ("Biến đổi ở bước mô hình", "log1p + StandardScaler (chưa áp dụng)")]
right = [("Dòng raw", len(raw)), ("Dòng mua giữ lại", len(purchases)), ("Dòng chứng từ C (bảng riêng)", len(returns)),
         ("Khách có mã", raw["CustomerID"].nunique()), ("Khách FS1 (có mua)", len(rfm)),
         ("Khách không có dòng mua", len(no_buy))]
for i, (a, b) in enumerate(left, start=5): ws[f"A{i}"], ws[f"B{i}"] = a, b
for i, (a, b) in enumerate(right, start=5): ws[f"D{i}"], ws[f"E{i}"] = a, b

def table(r, title, headers, data, fmt=None):
    ws.cell(r, 1, title).font = B; r += 1
    for j, h in enumerate(headers, start=1): ws.cell(r, j, h).font = B
    ws.row_dimensions[r].height = 25
    for i, row in enumerate(data, start=r + 1):
        for j, v in enumerate(row, start=1):
            v = v.item() if hasattr(v, "item") else v
            c = ws.cell(i, j, v)
            if fmt and isinstance(v, float): c.number_format = fmt
    return r + len(data) + 2

NHAN_XET = ["", "Không phân cụm được theo khách; không khôi phục được", "Giữ để làm feature trả hàng (FS3)",
            "Chỉ 40 dòng, giá trị 0"]
r = table(12, "1. Đối soát số dòng", [*accounting.columns, "Nhận xét"],
          [[*row, NHAN_XET[i]] for i, row in enumerate(accounting.itertuples(index=False))], "0.00")
rules = [
    ("Thiếu CustomerID", "Loại", "Loại", int(raw["CustomerID"].isna().sum()), "Không gán được cho khách"),
    ("Chứng từ C (hủy/trả)", "Loại", "Tách bảng riêng", len(returns), "Giữ thông tin trả hàng"),
    ("Quantity/UnitPrice ≤ 0 (không C)", "Loại", "Loại", int(len(rest) - len(purchases)), "Điều chỉnh kho, giá trị 0"),
    ("Dòng trùng hoàn toàn", "Loại", "Giữ", n_dup, "Không ảnh hưởng Frequency; tác động nhỏ tới Monetary"),
    ("Mã dịch vụ (POST, M, C2…)", "Giữ", "Giữ", n_svc, "Là doanh thu thật; loại khỏi feature sản phẩm ở FS2"),
    ("Ngoại lai", "Giữ", "Giữ", 0, "Không xóa dòng; xử lý ở bước biến đổi"),
]
r = table(r, "2. Quy tắc P-keep", ["Nhóm", "Baseline", "P-keep", "Số dòng", "Lý do"], rules)
r = table(r, "3. Thống kê feature FS1", ["Feature", *summary.columns],
          [[n, *vals] for n, vals in zip(summary.index, summary.itertuples(index=False))], "0.00")
r = table(r, "4. Đối chiếu baseline (mức dữ liệu)", ["Chỉ tiêu", "Baseline-policy", "FS1"],
          [[k, *v] for k, v in compare.iterrows()], "0.00")
for col, w in zip("ABCDEFGHIJ", [36, 22, 22, 22, 18, 16, 16, 44, 13, 13]): ws.column_dimensions[col].width = w
ws.freeze_panes = "A5"
wb.save("Working_details_FS1_TienXuLy.xlsx")
print("Đã lưu: data/customer_features.csv, Working_details_FS1_TienXuLy.xlsx")

Đã lưu: data/customer_features.csv, Working_details_FS1_TienXuLy.xlsx


**Nhận xét:** Đã lưu `data/customer_features.csv` (bảng FS1 cho bước mô hình) và workbook nhật ký `Working_details_FS1_TienXuLy.xlsx` theo bố cục mẫu của Hiếu (tiêu đề, trạng thái, thiết lập, các bảng kết quả, cột nhận xét).